In [1]:
topics = {
"digital_payments_processing": [
"payments",
"transactions",
"card",
"mobile",
"merchant",
"wallet",
"transfers",
"cross-border",
"fees",
"gateway"
],
"insurance_insurtech_underwriting": [
"insurance",
"insurtech",
"policy",
"broker",
"claims",
"coverage",
"health",
"underwriting",
"life",
"premium"
],
"blockchain_cryptocurrency_assets": [
"blockchain",
"crypto",
"wallet",
"tokens",
"decentralized",
"ethereum",
"custody",
"staking",
"trading",
"dapps"
],
"lending_credit_financing": [
"lending",
"loans",
"credit",
"interest",
"financing",
"capital",
"repayment",
"debt",
"underwriting",
"borrower"
],
"investment_wealth_trading": [
"investment",
"funds",
"equity",
"stocks",
"marketplace",
"etf",
"wealth",
"bonds",
"yield",
"portfolio"
],
"corporate_treasury_accounting": [
"cash",
"flow",
"liquidity",
"billing",
"invoices",
"expenses",
"treasury",
"accounting",
"payroll",
"sme"
],
"sustainability_carbon_climate": [
"carbon",
"credits",
"sustainability",
"esg",
"climate",
"emissions",
"offset",
"environmental",
"reporting",
"regenerative"
],
"tax_legal_planning": [
"tax",
"returns",
"filing",
"refund",
"income",
"legal",
"advisor",
"wills",
"provision",
"testament"
],
"mergers_acquisitions_advisory": [
"m&a",
"acquisitions",
"succession",
"exit",
"fundraising",
"advisory",
"valuation",
"transaction",
"equity",
"diligence"
]
}

In [2]:
# create list of keys

keys = list(topics.keys())
keys

['digital_payments_processing',
 'insurance_insurtech_underwriting',
 'blockchain_cryptocurrency_assets',
 'lending_credit_financing',
 'investment_wealth_trading',
 'corporate_treasury_accounting',
 'sustainability_carbon_climate',
 'tax_legal_planning',
 'mergers_acquisitions_advisory']

In [3]:
# OpenAI Agents SDK with Groq for categorization
# Install required packages: pip install openai-agents openai-agents[litellm] httpx

from groq import Groq

# Get Groq API key
with open(r'groq.txt', 'r', encoding='utf-8') as file:
    groq_api_key = str(file.readline()).strip()

import os
os.environ["GROQ_API_KEY"] = groq_api_key


In [4]:
# Define Pydantic models for structured output


system_prompt = """You are an expert in categorizing company descriptions into predefined topic categories.

You will receive:
1. A company description.
2. A list of topic categories with associated keywords.

Your task is to determine which topic(s) the company belongs to based strictly and exclusively on the provided company description.

------------------------------------------------------------
CRITICAL RULES (MUST FOLLOW)
------------------------------------------------------------

1. Use only the provided company description.
   - Do NOT use external knowledge about the company.
   - Do NOT rely on prior knowledge.
   - Do NOT assume missing facts.

2. Evidence-based classification only.
   - A topic may only be assigned if the description contains:
     • Explicit keywords from the topic list, OR
     • Very clear semantic equivalents.
   - If evidence is weak, implied, or ambiguous, do NOT assign the topic.

3. No speculation.
   - If the description does not clearly support a topic, exclude it.
   - If no topics are clearly supported, return an empty list.

4. Multi-label classification is allowed.
   - A company may belong to multiple topics if clearly supported by evidence.

5. Be conservative.
   - When in doubt, exclude the topic.

------------------------------------------------------------
TOPIC LIST
------------------------------------------------------------

{
  "digital_payments_processing": ["payments","transactions","card","mobile","merchant","wallet","transfers","cross-border","fees","gateway"],
  "insurance_insurtech_underwriting": ["insurance","insurtech","policy","broker","claims","coverage","health","underwriting","life","premium"],
  "blockchain_cryptocurrency_assets": ["blockchain","crypto","wallet","tokens","decentralized","ethereum","custody","staking","trading","dapps"],
  "lending_credit_financing": ["lending","loans","credit","interest","financing","capital","repayment","debt","underwriting","borrower"],
  "investment_wealth_trading": ["investment","funds","equity","stocks","marketplace","etf","wealth","bonds","yield","portfolio"],
  "corporate_treasury_accounting": ["cash","flow","liquidity","billing","invoices","expenses","treasury","accounting","payroll","sme"],
  "sustainability_carbon_climate": ["carbon","credits","sustainability","esg","climate","emissions","offset","environmental","reporting","regenerative"],
  "tax_legal_planning": ["tax","returns","filing","refund","income","legal","advisor","wills","provision","testament"],
  "mergers_acquisitions_advisory": ["m&a","acquisitions","succession","exit","fundraising","advisory","valuation","transaction","equity","diligence"]
}

------------------------------------------------------------
INTERNAL CLASSIFICATION METHOD (DO NOT OUTPUT THIS)
------------------------------------------------------------

Step 1: Scan the description for explicit keyword matches.
Step 2: Identify strong semantic equivalents where clearly justified.
Step 3: Verify the evidence directly supports the topic definition.
Step 4: Exclude weak, assumed, or indirect matches.
Step 5: Produce final JSON output only.

------------------------------------------------------------
REQUIRED OUTPUT FORMAT (STRICT JSON ONLY)
------------------------------------------------------------

If one or more topics apply:

{
  "assigned_topics": [
    {
      "topic": "topic_name",
      "evidence": "Exact quote or clear phrase from the description that justifies classification"
    }
  ]
}

If no topics apply:

{
  "assigned_topics": []
}

Do NOT include explanations outside the JSON output.
"""

company_description = """Upvest, founded in 2017 and based in Berlin, is a fintech startup that empowers financial institutions to integrate investment products into their applications through a modular and scalable API. The company operates in the embedded finance market, providing a comprehensive solution that includes regulatory licenses, making it easier for businesses to offer investment opportunities to their users. Upvest's business model revolves around offering its API as a service, generating revenue through subscription fees and transaction-based pricing. The company serves a diverse range of clients, including banks, fintech companies, and other financial institutions, helping them to enhance their product offerings and improve user engagement. Upvest is supported by the European Union and the European Union Regional Development Fund, and it is also involved in research and development projects aimed at improving blockchain transaction fee prediction tools. The startup's team comprises experts from renowned institutions such as Transferwise, Onfido, Klarna, Goldman Sachs, and UBS, ensuring a high level of expertise in capital markets, compliance, security, and technology.Keywords: Investment API, embedded finance, fintech, digital assets, regulatory licenses, scalable solutions, capital markets, blockchain, financial institutions, Berlin startup."""

user_prompt = "Here is the company description:" + company_description




In [5]:


client = Groq()
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
      {
        "role": "system",
        "content": system_prompt
      },
      {
        "role": "user",
        "content": user_prompt
      }
    ],
    temperature=1,
    max_completion_tokens=8192,
    top_p=1,
    reasoning_effort="medium",
    stream=False,
    response_format={"type": "json_object"},
    stop=None
)

result =completion.choices[0].message


In [6]:
result.content

'{"assigned_topics":[{"topic":"investment_wealth_trading","evidence":"integrate investment products into their applications"},{"topic":"blockchain_cryptocurrency_assets","evidence":"research and development projects aimed at improving blockchain transaction fee prediction tools"}]}'

In [7]:
import json
result_json = json.loads(result.content)


In [8]:
result_json['assigned_topics']

[{'topic': 'investment_wealth_trading',
  'evidence': 'integrate investment products into their applications'},
 {'topic': 'blockchain_cryptocurrency_assets',
  'evidence': 'research and development projects aimed at improving blockchain transaction fee prediction tools'}]

In [9]:
outputlist = []

for i in range(len(result_json['assigned_topics'])):
    outputlist.append(result_json['assigned_topics'][i]['topic'])

outputlist

['investment_wealth_trading', 'blockchain_cryptocurrency_assets']

In [10]:
scorelist = []

for i in range(len(keys)):
    if keys[i] in outputlist:
        scorelist.append(1)
    else:
        scorelist.append(0)

scorelist

[0, 0, 1, 0, 1, 0, 0, 0, 0]

In [11]:
import pandas as pd

# read excel

fintech = pd.read_excel(r'fintechdf.xlsx', sheet_name='Sheet1')

In [12]:
companynamelist = fintech['Name'].tolist()

companynamelist[0]

'Upvest'

In [ ]:
import time

# execute categorization loop for all companies

scoremetalist = []

for j in range(0,len(companynamelist)):
    company = companynamelist[j]
    company_description = str(fintech['Long description'][j])
    user_prompt = "Company name: " + company + "\nHere is the company description: " + company_description
    
    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
          {
            "role": "system",
            "content": system_prompt
          },
          {
            "role": "user",
            "content": user_prompt
          }
        ],
        temperature=1,
        max_completion_tokens=8192,
        top_p=1,
        reasoning_effort="medium",
        stream=False,
        response_format={"type": "json_object"},
        stop=None
    )
    
    result =completion.choices[0].message
    result_json = json.loads(result.content)
    
    outputlist = []
    
    for i in range(len(result_json['assigned_topics'])):
        outputlist.append(result_json['assigned_topics'][i]['topic'])
    
    scorelist = []
    
    for i in range(len(keys)):
        if keys[i] in outputlist:
            scorelist.append(1)
        else:
            scorelist.append(0)
    
    scoremetalist.append(scorelist)

    time.sleep(20)
    print("Company" + str(j) + "processed")

Company0processed
Company1processed
Company2processed
Company3processed
Company4processed


In [ ]:
# create dataframe from scoremetalist

scores = pd.DataFrame(scoremetalist, columns=keys)

# add scores to fintech dataframe
merged = pd.concat([fintech, scores], axis=1)






   

In [ ]:
merged.head()

,ID,Name,Long description,HQ city,Total funding (EUR M),Last round,Launch year,Founders,digital_payments_processing,insurance_insurtech_underwriting,blockchain_cryptocurrency_assets,lending_credit_financing,investment_wealth_trading,corporate_treasury_accounting,sustainability_carbon_climate,tax_legal_planning,mergers_acquisitions_advisory
0,961533,Upvest,"Upvest, founded in 2017 and based in Berlin, i...",Berlin,180.45,SERIES C,2017.0,Martin Kassing;Tobias Auferoth;Juha Ristolaine...,0,0,1,0,1,0,0,0,0
1,4121549,Ravio,Ravio.com is a cutting-edge startup that opera...,London,20.00,SERIES A,2022.0,Raymond Siems;Merten Wulfert;Roy Blanga;Evan M...,0,0,0,0,0,0,0,0,0
2,3813143,NG.CASH,NG.Cash is a digital banking platform that is ...,Rio de Janeiro,45.01,SERIES B,2021.0,Petrus Ballhausen Arruda;Luis Felipe Carvalho;...,1,0,0,0,0,0,0,0,0
3,3042,Skrill,Skrill is a digital wallet provider that facil...,Birmingham,NaN,ACQUISITION,2001.0,Benjamin Kullmann;Daniel Klein,1,0,1,0,0,0,0,0,0
4,4511900,Blockbrain,The Block Brain is a comprehensive cryptocurre...,Stuttgart,2.50,SEED,2022.0,Honza Ngo,0,0,1,0,1,0,0,0,0


In [ ]:
# export to excel

merged.to_excel(r'fintechdf_categorized.xlsx', index=False)